# Milestone 5 lab, notebook 6 of 8: Fix one day

## Where we are

| Notebook | What you do |
|---|---|
| 1. Did it run? | Read Airflow's run history and the pipeline's own. Run one night twice |
| 2. On time, as a number | Measure freshness. Turn "on time" into a number with a target |
| 3. Five signals | Monitoring asks if the job ran. Observability asks if the data is right |
| 4. Tell the person who can act | Airflow calls back, and the right person gets an alert |
| 5. Too many alerts | Count the careless way. Then group, send once, rank and silence |
| **6. Fix one day** (you are here) | Fix the source, run one night again, prove the number |
| 7. Four drills | Four faults on a clean day, each caught and repaired. Write an incident note |
| 8. The Monday answer | The whole week, the recovery, and the answer to the question |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. Airflow's page is http://localhost:8080 .

**About ten minutes of reading, and one minute of waiting** while Airflow runs one night twice.

In [1]:
import pandas as pd
from lab_helpers import (
    LAST_NIGHT, MON, airflow_tasks, airflow_tools, alerts_log, alerts_of, backfill, gate_results,
    manager_line, prepare_nights, proof, set_aside, sql,
)

prepare_nights(LAST_NIGHT)

The bonus week ran under Airflow: 7 nights, Mon 08-24 to Sun 08-30. 1 of 6 stores can be paid.


## Step 1: An incident with no failed task

**Tomas Reyes** runs store `S03`. On Monday 24 August his tills took **$777.20**. The pipeline published **$786.69**: a refund of $9.49 came in with a plus sign instead of a minus sign, so the gate set it aside, and the day is $9.49 too high.

Ask Airflow about Monday night:

In [2]:
airflow_tasks(MON)

,task,state,started (UTC),seconds
0,stores_send_files,success,15:33:57,0.6
1,load,success,15:33:57,0.5
2,gate_landing,success,15:33:58,1.5
3,clean,success,15:34:01,0.3
4,gate_clean,success,15:34:02,2.2
5,build,success,15:34:04,0.2
6,gate_publish,success,15:34:04,1.6
7,publish,success,15:34:07,0.3


All green. And still, for one store manager, the data was wrong. That is an incident.

An **incident** is a time when the data was wrong or late for the people who use it. It is not the same as a failed task. A task can fail at 02:00, be run again at 02:05, and hurt nobody. A task can be green while a manager's number is wrong. The question is who was hurt, and for how long.

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, what happens when it fails, and who is told.

## Step 2: Four questions

Every incident in this project is read with the same four questions.

**Which gate caught it?**

In [3]:
found = gate_results(MON)
found[found["test_name"].isin(["refund_gives_money_back", "line_adds_up"])]

,gate,test_name,action,status,bad_rows,title
0,clean,line_adds_up,quarantine,fail,1,A line adds up
1,clean,refund_gives_money_back,quarantine,fail,1,A refund gives money back


Two tests at the clean gate. Their action is **quarantine**: set the row aside. **What was set aside?**

In [4]:
set_aside("S03", MON)

,why,lines,cents
0,line_adds_up,1,949
1,refund_gives_money_back,1,949


One line, $9.49, for two reasons. **What did the manager see?**

In [5]:
manager_line("S03", MON)

S03  Tomas Reyes  net sales $786.69  Pay? DO NOT PAY YET
     Mon 08-24: 1 row set aside ($9.49): A line adds up; A refund gives money back


**Who was told?**

In [6]:
alerts_of(MON, about="S03")

,event,level,to_whom,title
0,new,ticket,Head of retail operations,"S03 Mon 08-24: Rows were set aside, and the day does not add up"


A ticket to the head of retail operations, on Monday night.

## Step 3: The root cause

The **root cause** is the first thing that went wrong, the one that made the others happen. "A check failed" is not a root cause: the check is how you found out. Ask "why?" until the answer is something a person can change.

Here is the row, as the store sent it:

In [7]:
sql("""
    SELECT line_id, receipt_id, line_type, quantity, unit_price_cents, amount_cents, refund_of_receipt_id
    FROM raw.sales_lines
    WHERE line_id = 'S03-0824-000008'
""")

,line_id,receipt_id,line_type,quantity,unit_price_cents,amount_cents,refund_of_receipt_id
0,S03-0824-000008,S03-T1-004493,refund,-1.000,949,949,S03-T2-004457


- Why was the number $9.49 too high? The refund row was set aside, so the $9.49 given back to a customer is not counted at all.
- Why was it set aside? Its `quantity` is -1, as a refund's should be. But its `amount_cents` is 949: positive. The row contradicts itself.
- Why? The store's export program wrote the amount without its minus sign.

The root cause is at the store: the export program dropped the sign of the amount. So that is where it is fixed.

## Step 4: Fix the source, never the warehouse

You could change the row in the warehouse by hand. Don't. The file in the landing folder is still wrong, and so is every copy of the night that loads it. A row repaired by hand comes back wrong.

Here "the store sends a corrected file" is one command. It runs inside the Airflow container, where Airflow's stores live:

In [8]:
print(airflow_tools.stores("fix", "--case", "1"))

Fixed at the source: case 1, A refund stored as a positive amount.
Deliver the day again to get the corrected files.


The store's corrected file will be sent the next time Monday's files are delivered: when Monday's night runs again.

## Step 5: Backfill one night

A **backfill** is running the pipeline again for past days, with one command. In Airflow 3 it is:

```sh
docker compose exec airflow airflow backfill create --dag-id lotus_nightly_m5 \
    --from-date 2026-08-24 --to-date 2026-08-25 --reprocess-behavior completed --max-active-runs 1
```

- `--from-date`, `--to-date`: the nights. Airflow stamps a night's run at 02:00 in Lotus's time zone, so the range ends the morning after.
- `--reprocess-behavior completed`: run these nights again, although they already ran. Without it, Airflow skips a night that has a run.
- `--max-active-runs 1`: one night at a time, oldest first.

Its opposite is running the whole week again: slow, and it hides what changed. Here only Monday was wrong, so only Monday runs. `backfill` types that command, and waits until the night has run. (It is `airflow_tools.backfill`, with a short wait when Airflow is still closing an earlier backfill.)

In [9]:
pd.DataFrame(backfill("2026-08-24", "2026-08-24"))

,day,state
0,2026-08-24,success


## Step 6: Reprocessing

What did the load step do this time? The pipeline's run history says:

In [10]:
sql("""
    SELECT run_day, step, started_at, detail
    FROM ops.run_history
    WHERE run_day = ? AND step = 'load'
    ORDER BY started_at DESC
    LIMIT 1
""", (MON,))

,run_day,step,started_at,detail
0,2026-08-24,load,2026-11-25 15:36:46.869391,"load: 1 new files, 113 rows copied to the raw tables. 1 changed file loaded again."


"1 changed file loaded again." The load step keeps a **fingerprint** of every file it loaded: a short code made from the file's bytes. The same bytes always give the same code. Tonight `S03`'s Monday file had the same name and a different fingerprint. So its old rows were removed, and the file was loaded again.

**Reprocessing** is loading and building the same data again, after the source was corrected. Is the file there once, or twice?

In [11]:
sql("""
    SELECT file_name, row_count, received_at, checksum AS fingerprint
    FROM raw.loaded_files
    WHERE file_name = 'sales_lines_S03_2026-08-24.csv'
""")

,file_name,row_count,received_at,fingerprint
0,sales_lines_S03_2026-08-24.csv,113,2026-08-24 23:18:00,b091744cd4f43aaa055ed95f719a57d756063bdd


Once, with its new fingerprint.

## Step 7: The proof

The published number, beside the answer key. Nobody's pipeline code made the answer key: it is added up from the stores' own receipts.

In [12]:
proof("S03", MON)

,store,day,published,answer key,equal,status
0,S03,Mon 08-24,$777.20,$777.20,True,ok


In [13]:
manager_line("S03", MON)

S03  Tomas Reyes  net sales $777.20  Pay? yes, bonus $7.77


**$777.20**, equal to the answer key, and the status is `ok`. Tomas's ticket:

In [14]:
alerts_of(MON, about="S03")

,event,level,to_whom,title
0,new,ticket,Head of retail operations,"S03 Mon 08-24: Rows were set aside, and the day does not add up"
1,resolved,ticket,Head of retail operations,"S03 Mon 08-24: Rows were set aside, and the day does not add up"


In [15]:
alerts_log(1)

2026-08-25 02:00  RESOLVED to Head of retail operations: S03 Mon 08-24: Rows were set aside, and the day does not add up


`RESOLVED`, sent to the head of retail operations. The log line carries the night's own clock, 02:00 on 25 August, because the backfill ran that night.

Nothing was "taken out of quarantine" by hand. The clean tables and the quarantine are built again from the raw rows every night. The raw row is right now, so no test returns it, and it is simply not set aside.

## Step 8: A safe re-run

One last question. What if somebody runs Monday once more, by mistake?

In [16]:
published_before = sql("SELECT store_id, sales_date, net_sales_cents FROM output.daily_store_sales ORDER BY 1, 2")
log_before = len(airflow_tools.alerts_log(1000).splitlines())
pd.DataFrame(airflow_tools.run_night("2026-08-24"))

,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [17]:
published_after = sql("SELECT store_id, sales_date, net_sales_cents FROM output.daily_store_sales ORDER BY 1, 2")
log_after = len(airflow_tools.alerts_log(1000).splitlines())
print("Published numbers are the same:", published_before.equals(published_after))
print("New lines in the alerts log:   ", log_after - log_before)
print(sql("SELECT detail FROM ops.run_history WHERE step = 'load' ORDER BY started_at DESC LIMIT 1").loc[0, "detail"])

Published numbers are the same: True
New lines in the alerts log:    0
load:         0 new files, 0 rows copied to the raw tables.


Nothing changed. The load step found 0 new files. The published numbers are equal. No alert was sent twice.

A **safe re-run** is a run that can be repeated and gives the same result: nothing is counted twice. It is what makes recovery boring, and boring is the goal. Its opposite is a load that adds the same rows again on each run: the second run would double Tomas's Monday.

## What you now know

- An **incident** is a time when the data was wrong or late for the people who use it. Every task can be green during one.
- The **root cause** is the first thing that went wrong: here, an export program that dropped a minus sign. The check is how you found out.
- Fix at the source, never in the warehouse.
- **Reprocessing** loads a corrected file again: a new fingerprint means its old rows go first.
- A **backfill** runs past nights again with one command. Only the night that was wrong is run.
- A **safe re-run** gives the same result the second time.

**Say it in an interview:** "A refund with a lost minus sign made a manager's Monday $9.49 too high. No task failed; it was still an incident. The root cause was the store's export, so the store sent a corrected file. I ran an Airflow backfill for that one night: the load step saw a new fingerprint, replaced the file's rows, the number equalled the answer key at $777.20, and the ticket closed by itself. Running the night again changed nothing."

**The theory:** [Incremental Processing & Backfills](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/incremental-processing-and-backfills), and [Airflow: Backfills, Catchup, Retries & SLA](https://www.d8loop.com/modules/knowledge-corner/airflow/backfills-catchup-retries-and-sla).

Next: [notebook 7](07_four_drills.ipynb). Four new faults, on purpose.